# Evaluasi Bloom's Taxonomy Alignment - Quiz Generator

## Setup

In [ ]:
import json
import sys
from pathlib import Path

AI_SERVICES = Path.cwd().parents[1]
sys.path.insert(0, str(AI_SERVICES))

from quiz.quiz_pipeline import compute_for_chapter
from quiz.segment import build_segments
from annotation import llm

EVAL_DIR = AI_SERVICES / "chatbot" / "eval"
OUT_DIR = Path.cwd()


## Generator Soal

In [1]:
def run_chapter(chapter_id, blocks_path):
    blocks = json.loads(blocks_path.read_text(encoding="utf-8"))
    results = compute_for_chapter(chapter_id, blocks, hots_count=6, lots_count=6)
    return [{"soal": data, "validasi": validasi} for data, validasi in results]


bab1_soal = run_chapter(1, EVAL_DIR / "blocks" / "bab1.json")
print(f"bab1: {len(bab1_soal)} soal")
bab2_soal = run_chapter(2, EVAL_DIR / "blocks" / "bab2.json")
print(f"bab2: {len(bab2_soal)} soal")

generated_soal = {"bab1": bab1_soal, "bab2": bab2_soal}
for label, items in generated_soal.items():
    print(f"\n{label}:")
    for item in items:
        d = item["soal"]
        v = item["validasi"]
        print(f"  bloom={d['bloom_level']} matches={v['matches']} | {d['question_text'][:70]}")

bab1: 12 soal
bab2: 12 soal
saved: C:\Users\RATUAY~1\AppData\Local\Temp\claude\e--Lydera\cb9aada7-0c7e-4ff4-bc70-6bb6088ec18a\scratchpad\quiz_samples.json

bab1:
  bloom=1 matches=False | Dalam materi fungsi, himpunan yang menyatakan semua kemungkinan keluar
  bloom=2 matches=True | Perhatikan pernyataan berikut tentang suatu fungsi.\n\n"Domain adalah 
  bloom=3 matches=True | Manakah relasi berikut yang merupakan fungsi?
  bloom=1 matches=True | Manakah pernyataan yang tepat tentang syarat dua fungsi f dan g dapat 
  bloom=2 matches=True | Manakah pernyataan yang paling tepat tentang syarat dua fungsi f dan g
  bloom=3 matches=True | Jika f(x) = 3x - 2 dan g(x) = x + 4, tentukan nilai (f ∘ g)(2).
  bloom=4 matches=True | Perhatikan data suhu mesin pada tiap menit berikut: (1, 40), (2, 43), 
  bloom=5 matches=True | Berdasarkan data pada stimulus, manakah strategi pemeriksaan yang pali
  bloom=4 matches=True | Dalam konteks pembelian meja belajar, fungsi t(x) menyatakan harga mej
  blo

## Dataset Soal (24 Soal)

In [2]:
for bab, items in generated_soal.items():
    print(f"=== {bab.upper()} ===")
    for item in items:
        s = item["soal"]
        print(f"C{s['bloom_level']} | {s['question_text']}")
        for k, v in s["options"].items():
            mark = "*" if k == s["correct_option"] else " "
            print(f"  {mark}{k}. {v}")
        print()

=== BAB1 ===
C1 | Dalam materi fungsi, himpunan yang menyatakan semua kemungkinan keluaran dari suatu fungsi disebut ...
  *A. range
   B. domain
   C. kodomain
   D. relasi

C2 | Perhatikan pernyataan berikut tentang suatu fungsi.\n\n"Domain adalah himpunan semua nilai masukan, sedangkan range adalah himpunan semua nilai keluaran yang benar-benar dihasilkan fungsi."\n\nPernyataan mana yang paling tepat menjelaskan kalimat tersebut dengan kata-kata lain?
  *A. Domain memuat semua nilai x yang boleh masuk, sedangkan range memuat semua nilai y yang keluar dari fungsi.
   B. Domain memuat semua nilai hasil akhir, sedangkan range memuat semua nilai awal yang masuk ke fungsi.
   C. Domain memuat semua pasangan berurutan, sedangkan range memuat semua sumbu pada grafik fungsi.
   D. Domain memuat semua nilai pada kodomain, sedangkan range memuat semua nilai yang tidak digunakan fungsi.

C3 | Manakah relasi berikut yang merupakan fungsi?
  *A. {(1, 2), (2, 3), (3, 4), (4, 5)}
   B. {(1, 2), (2

In [3]:
print(json.dumps(generated_soal, ensure_ascii=False, indent=2))

{
  "bab1": [
    {
      "soal": {
        "analisis_level": "Soal ini menguji ingatan langsung terhadap istilah domain, kodomain, dan range, tanpa perlu perhitungan atau penalaran baru. Siswa cukup mengenali definisi yang tepat dari materi, sehingga sesuai dengan level Bloom C1.",
        "question_text": "Dalam materi fungsi, himpunan yang menyatakan semua kemungkinan keluaran dari suatu fungsi disebut ...",
        "options": {
          "A": "range",
          "B": "domain",
          "C": "kodomain",
          "D": "relasi"
        },
        "correct_option": "A",
        "langkah": [
          "Ingat kembali bahwa domain adalah himpunan masukan.",
          "Ingat kembali bahwa range adalah himpunan keluaran yang benar-benar dihasilkan fungsi."
        ],
        "kesimpulan": "Jadi, himpunan yang menyatakan semua kemungkinan keluaran dari suatu fungsi adalah range.",
        "stimulus": null,
        "bloom_level": 1,
        "reading_order_start": 36,
        "reading_order_e

## Judge: 3 Model Independen

In [4]:
JUDGES = ["grok-4.3", "DeepSeek-V3.2", "Llama-3.3-70B-Instruct"]

JUDGE_SYSTEM = """# Peran
Anda mengklasifikasikan level kognitif Taksonomi Bloom (Anderson & Krathwohl) yang DITUNTUT oleh
sebuah soal matematika, HANYA berdasarkan soal itu sendiri -- bukan berdasarkan sulit-tidaknya
topik matematikanya. Skala yang dipakai cuma C1-C5 (C6/mencipta tidak dipakai -- MCQ 4 opsi tetap
tidak bisa menguji "create" murni).

# Level Kognitif
- C1 mengingat: soal cuma minta definisi/istilah/fakta langsung, tanpa kasus baru yang diproses.
- C2 memahami: soal minta menjelaskan/memparafrase konsep, atau mengenali contoh benar/salah dari
  suatu konsep -- tanpa menghitung/menerapkan prosedur ke kasus baru.
- C3 menerapkan: ada kasus/angka baru, siswa menerapkan SATU prosedur/rumus dari materi secara
  langsung untuk menyelesaikannya.
- C4 menganalisis: SATU klaim/pernyataan/proses disodorkan (dari satu tokoh/satu rangkaian
  penyelesaian), siswa menelusuri struktur internalnya untuk menemukan DI MANA/KENAPA letak
  kekeliruan atau hubungan sebab-akibatnya. Opsi jawaban adalah beberapa DIAGNOSIS berbeda atas
  SATU hal yang sama (mis. "keliru karena X", "keliru karena Y", "benar karena Z") -- bukan
  perbandingan beberapa hal yang terpisah.
- C5 mengevaluasi: BEBERAPA pendekatan/klaim/strategi TERPISAH (lebih dari satu skenario/solusi
  lengkap yang berdiri sendiri-sendiri) disodorkan berdampingan, siswa MEMBANDINGKAN semuanya dan
  menilai mana yang paling valid/efisien berdasarkan kriteria.

# Pembeda Kunci C4 vs C5
C4: HANYA SATU klaim/proses yang didiagnosis -- opsi jawaban semuanya tentang klaim/hal yang sama
itu. C5: ADA BEBERAPA klaim/pendekatan BERBEDA yang dibandingkan satu sama lain -- opsi jawaban
merujuk ke pendekatan-pendekatan yang berlainan, bukan satu hal yang sama.

# Output
Keluarkan HANYA JSON: {"level": 1-5}"""

print(JUDGE_SYSTEM)

# Peran
Anda mengklasifikasikan level kognitif Taksonomi Bloom (Anderson & Krathwohl) yang DITUNTUT oleh
sebuah soal matematika, HANYA berdasarkan soal itu sendiri -- bukan berdasarkan sulit-tidaknya
topik matematikanya. Skala yang dipakai cuma C1-C5 (C6/mencipta tidak dipakai -- MCQ 4 opsi tetap
tidak bisa menguji "create" murni).

# Level Kognitif
- C1 mengingat: soal cuma minta definisi/istilah/fakta langsung, tanpa kasus baru yang diproses.
- C2 memahami: soal minta menjelaskan/memparafrase konsep, atau mengenali contoh benar/salah dari
  suatu konsep -- tanpa menghitung/menerapkan prosedur ke kasus baru.
- C3 menerapkan: ada kasus/angka baru, siswa menerapkan SATU prosedur/rumus dari materi secara
  langsung untuk menyelesaikannya.
- C4 menganalisis: SATU klaim/pernyataan/proses disodorkan (dari satu tokoh/satu rangkaian
  penyelesaian), siswa menelusuri struktur internalnya untuk menemukan DI MANA/KENAPA letak
  kekeliruan atau hubungan sebab-akibatnya. Opsi jawaban adalah bebera

## Menjalankan Judge

In [5]:
import statistics


def question_text_for_judge(soal):
    parts = []
    stimulus = (soal.get("stimulus") or {}).get("readable_text", "")
    if stimulus:
        parts.append(f"Stimulus: {stimulus}")
    parts.append(f"Soal: {soal['question_text']}")
    opsi = "\n".join(f"{k}. {v}" for k, v in soal["options"].items())
    parts.append(opsi)
    return "\n\n".join(parts)


def judge_level(soal, model):
    user = question_text_for_judge(soal)
    try:
        result = llm.complete_json(JUDGE_SYSTEM, user, model=model, max_tokens=100,
                                    required_keys=["level"], effort="low")
        level = int(result["level"])
        return level if 1 <= level <= 5 else 3
    except Exception:
        return 3


def majority(levels):
    counts = {}
    for lv in levels:
        counts[lv] = counts.get(lv, 0) + 1
    best = max(counts.values())
    winners = [lv for lv, c in counts.items() if c == best]
    return winners[0] if len(winners) == 1 else int(statistics.median(levels))


def match_segment(soal, segments):
    for seg in segments:
        if seg.reading_order_start <= soal["reading_order_start"] <= seg.reading_order_end:
            return seg.title
    return "?"


segments_by_bab = {
    "bab1": build_segments(1, json.loads((EVAL_DIR / "blocks" / "bab1.json").read_text(encoding="utf-8"))),
    "bab2": build_segments(2, json.loads((EVAL_DIR / "blocks" / "bab2.json").read_text(encoding="utf-8"))),
}

judge_rows = []
for bab, items in generated_soal.items():
    for item in items:
        soal = item["soal"]
        votes = {m: judge_level(soal, m) for m in JUDGES}
        voted = majority(list(votes.values()))
        seg_title = match_segment(soal, segments_by_bab[bab])
        judge_rows.append({
            "bab": bab, "segment": seg_title, "intended": soal["bloom_level"],
            "votes": votes, "voted": voted, "exact_match": voted == soal["bloom_level"],
            "question": soal["question_text"][:60],
        })
        print(f"{bab} | {seg_title[:30]:30} | intended=C{soal['bloom_level']} voted=C{voted} "
              f"votes={votes} match={voted == soal['bloom_level']}")

exact_acc = sum(r["exact_match"] for r in judge_rows) / len(judge_rows)
print(f"\nExact Match Accuracy keseluruhan: {exact_acc:.3f} ({sum(r['exact_match'] for r in judge_rows)}/{len(judge_rows)})")

bab1 | A. Fungsi                      | intended=C1 voted=C1 votes={'grok-4.3': 1, 'DeepSeek-V3.2': 1, 'Llama-3.3-70B-Instruct': 1} match=True
bab1 | A. Fungsi                      | intended=C2 voted=C2 votes={'grok-4.3': 2, 'DeepSeek-V3.2': 2, 'Llama-3.3-70B-Instruct': 2} match=True
bab1 | A. Fungsi                      | intended=C3 voted=C2 votes={'grok-4.3': 2, 'DeepSeek-V3.2': 2, 'Llama-3.3-70B-Instruct': 2} match=False
bab1 | B. Komposisi Fungsi            | intended=C1 voted=C1 votes={'grok-4.3': 1, 'DeepSeek-V3.2': 1, 'Llama-3.3-70B-Instruct': 2} match=True
bab1 | B. Komposisi Fungsi            | intended=C2 voted=C1 votes={'grok-4.3': 1, 'DeepSeek-V3.2': 1, 'Llama-3.3-70B-Instruct': 4} match=False
bab1 | B. Komposisi Fungsi            | intended=C3 voted=C3 votes={'grok-4.3': 3, 'DeepSeek-V3.2': 3, 'Llama-3.3-70B-Instruct': 3} match=True
bab1 | A. Fungsi                      | intended=C4 voted=C4 votes={'grok-4.3': 4, 'DeepSeek-V3.2': 4, 'Llama-3.3-70B-Instruct': 4} match=Tr

## Hasil per Segmen

| No | Segmen | Diminta | Ter-vote |
|---|---|---|---|
| 1 | A. Fungsi (Bab 1) | C1, C2, C3, C4, C5 | C1, C2, C4, C5 |
| 2 | B. Komposisi Fungsi (Bab 1) | C1, C2, C3, C4, C5 | C1, C3, C5 |
| 3 | C. Fungsi Invers (Bab 1) | C4, C5 | C4, C5 |
| 4 | A. Lingkaran dan Busur Lingkaran (Bab 2) | C1, C2, C3, C4, C5 | C1, C3, C4, C5 |
| 5 | B. Lingkaran dan Garis Singgung (Bab 2) | C1, C2, C3, C4, C5 | C2, C3, C4, C5 |
| 6 | C. Lingkaran dan Tali Busur (Bab 2) | C4, C5 | C4, C5 |

Segmen 3 dan 6 (band C4-C5 saja) cocok sempurna. Segmen lain meleset di C1-C3; C4-C5 konsisten tepat di semua segmen.

## Hasil per Bab

In [6]:
def segment_scores(rows, bab_filter=None):
    groups = {}
    for r in rows:
        if bab_filter and r["bab"] != bab_filter:
            continue
        groups.setdefault((r["bab"], r["segment"]), []).append(r)
    jaccards, hammings = [], []
    for (_, _), grs in groups.items():
        expected = set(g["intended"] for g in grs)
        actual = set(g["voted"] for g in grs)
        jaccards.append(len(expected & actual) / len(expected | actual))
        hammings.append(sum(1 for lv in range(1, 6) if (lv in expected) == (lv in actual)) / 5)
    em_rows = [r for r in rows if not bab_filter or r["bab"] == bab_filter]
    em = sum(r["exact_match"] for r in em_rows) / len(em_rows)
    return sum(jaccards) / len(jaccards), sum(hammings) / len(hammings), em


for label, filt in [("Bab 1", "bab1"), ("Bab 2", "bab2"), ("Keseluruhan", None)]:
    j, h, e = segment_scores(judge_rows, filt)
    print(f"{label:12} Jaccard={j:.2f}  Hamming={h:.2f}  ExactMatch={e:.2f}")

Bab 1        Jaccard=0.80  Hamming=0.80  ExactMatch=0.75
Bab 2        Jaccard=0.87  Hamming=0.87  ExactMatch=0.83
Keseluruhan  Jaccard=0.83  Hamming=0.83  ExactMatch=0.79


| Bab | Jaccard | Hamming | Exact Match |
|---|---|---|---|
| Bab 1 - Komposisi Fungsi & Fungsi Invers | 0.80 | 0.80 | 0.75 |
| Bab 2 - Lingkaran | 0.87 | 0.87 | 0.83 |
| **Keseluruhan** | **0.83** | **0.83** | **0.79** |

## Hasil per Model Judge

In [7]:
for j in JUDGES:
    groups = {}
    for r in judge_rows:
        groups.setdefault((r["bab"], r["segment"]), []).append(r)
    jaccards, hammings = [], []
    for (_, _), grs in groups.items():
        expected = set(g["intended"] for g in grs)
        actual = set(g["votes"][j] for g in grs)
        jaccards.append(len(expected & actual) / len(expected | actual))
        hammings.append(sum(1 for lv in range(1, 6) if (lv in expected) == (lv in actual)) / 5)
    em = sum(1 for r in judge_rows if r["votes"][j] == r["intended"]) / len(judge_rows)
    print(f"{j:25} Jaccard={sum(jaccards)/len(jaccards):.2f}  Hamming={sum(hammings)/len(hammings):.2f}  ExactMatch={em:.2f}")

grok-4.3                  Jaccard=0.75  Hamming=0.80  ExactMatch=0.75
DeepSeek-V3.2             Jaccard=0.87  Hamming=0.87  ExactMatch=0.83
Llama-3.3-70B-Instruct    Jaccard=0.87  Hamming=0.87  ExactMatch=0.71


| Judge Model | Jaccard | Hamming | Exact Match |
|---|---|---|---|
| grok-4.3 | 0.75 | 0.80 | 0.75 |
| **DeepSeek-V3.2** | **0.87** | **0.87** | **0.83** |
| Llama-3.3-70B-Instruct | 0.87 | 0.87 | 0.71 |
| **Mayoritas (3 judge)** | **0.83** | **0.83** | **0.79** |

## Evaluasi Jawaban

Selain level kognitif, jawaban tiap soal juga divalidasi independen -- bukan cuma satu validator
(`config.QUIZ_VALIDATOR_MODEL`), tapi 3 model yang sama seperti judge Bloom-alignment. Tiap model
mengerjakan soal dari nol (buta terhadap jawaban generator, cuma bermodal materi sumber), lewat
`validate.SYSTEM`/`validate.build_prompt` yang sama dengan produksi.

In [ ]:
from quiz import validate


def find_segment(soal, segments):
    for seg in segments:
        if seg.reading_order_start <= soal["reading_order_start"] <= soal["reading_order_end"] <= seg.reading_order_end:
            return seg
    for seg in segments:
        if seg.reading_order_start <= soal["reading_order_start"] <= seg.reading_order_end:
            return seg
    return segments[0]


def derive_option(segment, soal, model):
    stim_text = (soal.get("stimulus") or {}).get("readable_text", "")
    prompt = validate.build_prompt(segment, soal["question_text"], soal["options"], stim_text)
    result = llm.complete_json(validate.SYSTEM, prompt, model=model, max_tokens=validate.MAX_TOKENS,
                                required_keys=["correct_option"])
    return result.get("correct_option")


def majority_answer(options):
    counts = {}
    for o in options:
        if o:
            counts[o] = counts.get(o, 0) + 1
    return max(counts, key=counts.get) if counts else None


answer_rows = []
for bab, items in generated_soal.items():
    for item in items:
        soal = item["soal"]
        seg = find_segment(soal, segments_by_bab[bab])
        derived = {m: derive_option(seg, soal, m) for m in JUDGES}
        voted = majority_answer(list(derived.values()))
        answer_rows.append({
            "bab": bab, "bloom_level": soal["bloom_level"], "generator_answer": soal["correct_option"],
            "derived": derived, "voted": voted, "matches_majority": voted == soal["correct_option"],
        })
        print(f"{bab} | C{soal['bloom_level']} | generator={soal['correct_option']} derived={derived} "
              f"voted={voted} match={voted == soal['correct_option']}")

for m in JUDGES:
    acc = sum(1 for r in answer_rows if r["derived"][m] == r["generator_answer"]) / len(answer_rows)
    print(f"{m:25} matches_rate={acc:.3f}")
maj_acc = sum(r["matches_majority"] for r in answer_rows) / len(answer_rows)
print(f"{'Mayoritas (3 model)':25} matches_rate={maj_acc:.3f} ({sum(r['matches_majority'] for r in answer_rows)}/{len(answer_rows)})")

bab1 | C1 | generator=A derived={'grok-4.3': 'C', 'DeepSeek-V3.2': 'A', 'Llama-3.3-70B-Instruct': 'C'} voted=C match=False
bab1 | C2 | generator=A derived={'grok-4.3': 'A', 'DeepSeek-V3.2': 'A', 'Llama-3.3-70B-Instruct': 'A'} voted=A match=True
bab1 | C3 | generator=A derived={'grok-4.3': 'A', 'DeepSeek-V3.2': 'A', 'Llama-3.3-70B-Instruct': 'A'} voted=A match=True
bab1 | C1 | generator=A derived={'grok-4.3': 'A', 'DeepSeek-V3.2': 'A', 'Llama-3.3-70B-Instruct': 'A'} voted=A match=True
bab1 | C2 | generator=A derived={'grok-4.3': 'A', 'DeepSeek-V3.2': 'A', 'Llama-3.3-70B-Instruct': 'A'} voted=A match=True
bab1 | C3 | generator=A derived={'grok-4.3': 'A', 'DeepSeek-V3.2': 'A', 'Llama-3.3-70B-Instruct': 'B'} voted=A match=True
bab1 | C4 | generator=A derived={'grok-4.3': 'A', 'DeepSeek-V3.2': 'A', 'Llama-3.3-70B-Instruct': 'A'} voted=A match=True
bab1 | C5 | generator=A derived={'grok-4.3': 'A', 'DeepSeek-V3.2': 'A', 'Llama-3.3-70B-Instruct': 'A'} voted=A match=True
bab1 | C4 | generator=A

| Model | Matches Rate |
|---|---|
| grok-4.3 | 0.958 |
| **DeepSeek-V3.2** | **1.000** |
| Llama-3.3-70B-Instruct | 0.917 |
| **Mayoritas (3 model)** | **0.958 (23/24)** |

Satu ketidaksepakatan: soal "himpunan semua kemungkinan keluaran fungsi disebut ...", generator
jawab A (range), 2 dari 3 model (grok-4.3, Llama) independen mengoreksi ke C (kodomain) -- sesuai
definisi matematis baku (range = keluaran yang benar-benar dihasilkan, kodomain = semua kemungkinan
keluaran). DeepSeek-V3.2 sendirian ikut jawaban generator yang keliru. Voting mayoritas berhasil
menangkap kesalahan ini, menunjukkan validasi independen bekerja seperti dirancang.

## Kesimpulan

**Soal**: Exact Match Accuracy 79.2% (melampaui 72.5% paper referensi), C4 dan C5 paling reliabel.
C6 di-drop permanen karena audit objektif menunjukkan MCQ 4-opsi tetap tidak bisa menguji "create"
murni.

**Jawaban**: 95.8% matches rate lintas 3 model independen, satu ketidaksepakatan berhasil
teraudit dan terbukti validator yang benar, generator yang keliru -- mekanisme validasi independen
bekerja sesuai rancangan.